In [2]:
from datasets import load_dataset
dataset = load_dataset("rajpurkar/squad")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [3]:
#adding the eos
def add_end_of_text(example):
    example['question'] = example['question'] + '<|endoftext|>'
    return example

dataset = dataset.remove_columns(['id', 'title', 'context', 'answers'])
dataset = dataset.map(add_end_of_text)



In [4]:
#previewing the dataset

dataset['train']['question'][:10]

['To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|endoftext|>',
 'What is in front of the Notre Dame Main Building?<|endoftext|>',
 'The Basilica of the Sacred heart at Notre Dame is beside to which structure?<|endoftext|>',
 'What is the Grotto at Notre Dame?<|endoftext|>',
 'What sits on top of the Main Building at Notre Dame?<|endoftext|>',
 'When did the Scholastic Magazine of Notre dame begin publishing?<|endoftext|>',
 "How often is Notre Dame's the Juggler published?<|endoftext|>",
 'What is the daily student paper at Notre Dame called?<|endoftext|>',
 'How many student news papers are found at Notre Dame?<|endoftext|>',
 'In what year did the student paper Common Sense begin publication at Notre Dame?<|endoftext|>']

# Tokenization

In [ ]:
# normalization -> pre-tokenizing -> tokenizing -> passing to the model -> post processing

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = 'distilgpt2' #
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

In [22]:
#testing by converting a simple sentence to a tokenized sentence

sequence = "Adam is so cool!"
tokens = tokenizer.tokenize(sequence)
print(tokens)

['Adam', 'Ġis', 'Ġso', 'Ġcool', '!']


In [23]:
#testing by convertingto ids
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)


[23159, 318, 523, 3608, 0]


In [24]:
#the tokenizer is able to convert the tokens to ids and provide their attention mask
tokenizer(sequence)

{'input_ids': [23159, 318, 523, 3608, 0], 'attention_mask': [1, 1, 1, 1, 1]}

In [ ]:
#tokenizing the whole dataset
def tokenize_function(examples):
    return tokenizer(examples['question'],truncation=True)

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=['question']
    )

In [26]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 10570
    })
})

# Data Preprocessing

In [27]:
#splitting the dataset into equal chunks for proper processing without padding

block_size = 128

def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in
                            examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size

    # populate each of input_ids and other keys 
    result = {
        k: [t[i : i + block_size] for i in range(0,
            total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    # add labels because we'll need it as the output
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)

In [28]:
#previewing the tokens
print(lm_datasets['train']['input_ids'][0])

[2514, 4150, 750, 262, 5283, 5335, 7910, 1656, 287, 1248, 3365, 287, 406, 454, 8906, 4881, 30, 50256, 2061, 318, 287, 2166, 286, 262, 23382, 20377, 8774, 11819, 30, 50256, 464, 32520, 3970, 286, 262, 17380, 2612, 379, 23382, 20377, 318, 13970, 284, 543, 4645, 30, 50256, 2061, 318, 262, 10299, 33955, 379, 23382, 20377, 30, 50256, 2061, 10718, 319, 1353, 286, 262, 8774, 11819, 379, 23382, 20377, 30, 50256, 2215, 750, 262, 3059, 349, 3477, 11175, 286, 23382, 288, 480, 2221, 12407, 30, 50256, 2437, 1690, 318, 23382, 20377, 338, 262, 39296, 1754, 3199, 30, 50256, 2061, 318, 262, 4445, 3710, 3348, 379, 23382, 20377, 1444, 30, 50256, 2437, 867, 3710, 1705, 9473, 389, 1043, 379, 23382, 20377, 30, 50256, 818, 644, 614, 750, 262, 3710, 3348]


In [29]:
#previewing the dataset but understanding the token:


tokenizer.decode(lm_datasets['train']['input_ids'][0])

"To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?<|endoftext|>What is in front of the Notre Dame Main Building?<|endoftext|>The Basilica of the Sacred heart at Notre Dame is beside to which structure?<|endoftext|>What is the Grotto at Notre Dame?<|endoftext|>What sits on top of the Main Building at Notre Dame?<|endoftext|>When did the Scholastic Magazine of Notre dame begin publishing?<|endoftext|>How often is Notre Dame's the Juggler published?<|endoftext|>What is the daily student paper at Notre Dame called?<|endoftext|>How many student news papers are found at Notre Dame?<|endoftext|>In what year did the student paper"

In [30]:
#splitting the dataset into train and validation


small_train_dataset = \
    lm_datasets["train"].shuffle(seed=42).select(range(100))
small_eval_dataset = \
    lm_datasets["validation"].shuffle(seed=42).select(range(100))
    



In [31]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 5448.82it/s]


In [32]:
# Specifyign the training arguements adn setup the trainer
training_args = TrainingArguments(
    f"{model_checkpoint}-squad",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
)

trainer.train()



Epoch,Training Loss,Validation Loss
1,No log,3.695442
2,No log,3.521419
3,No log,3.462922


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.12s/it]
/opt/miniconda3/envs/lec4-new/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=39, training_loss=3.8676425249148636, metrics={'train_runtime': 173.016, 'train_samples_per_second': 1.734, 'train_steps_per_second': 0.225, 'total_flos': 9798628147200.0, 'train_loss': 3.8676425249148636, 'epoch': 3.0})

In [35]:
# calculating the perplexity
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
No log,3.462922,3


Perplexity: 31.91


In [36]:
tokenizer.save_pretrained("gpt2-squad")

('gpt2-squad/tokenizer_config.json', 'gpt2-squad/tokenizer.json')